In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

all_sheets = pd.read_excel('/content/drive/MyDrive/Cadetx /ev_charging_dataset.xlsx', sheet_name=None)
sessions_df = all_sheets['sessions']

print(sessions_df[['customer_id', 'user_type', 'total_cost']].head())

  customer_id user_type  total_cost
0  CUST_00134     fleet       26.74
1  CUST_00195  delivery       18.72
2  CUST_01935    public       14.52
3  CUST_00036    public       12.78
4  CUST_00011      taxi       17.75


In [3]:
customer_clv = sessions_df.groupby(['customer_id', 'user_type']).agg(
    frequency=('session_id', 'count'),
    avg_spend=('total_cost', 'mean'),
    total_spend=('total_cost', 'sum')
).reset_index()

print(customer_clv.head())
print(customer_clv.shape)

  customer_id user_type  frequency  avg_spend  total_spend
0  CUST_00001  delivery         68  24.924118      1694.84
1  CUST_00001     fleet         70  29.991714      2099.42
2  CUST_00001    public         53  26.519057      1405.51
3  CUST_00001      taxi         76  27.080132      2058.09
4  CUST_00002  delivery         65  28.033077      1822.15
(25847, 5)


In [4]:
customer_clv['clv'] = customer_clv['frequency'] * customer_clv['avg_spend']

print(customer_clv.head())

  customer_id user_type  frequency  avg_spend  total_spend      clv
0  CUST_00001  delivery         68  24.924118      1694.84  1694.84
1  CUST_00001     fleet         70  29.991714      2099.42  2099.42
2  CUST_00001    public         53  26.519057      1405.51  1405.51
3  CUST_00001      taxi         76  27.080132      2058.09  2058.09
4  CUST_00002  delivery         65  28.033077      1822.15  1822.15


In [5]:
segment_clv = customer_clv.groupby('user_type').agg(
    avg_clv=('clv', 'mean'),
    median_clv=('clv', 'median'),
    total_clv=('clv', 'sum'),
    customer_count=('customer_id', 'count')
).reset_index()

print(segment_clv)

  user_type     avg_clv  median_clv   total_clv  customer_count
0  delivery  314.198639     153.720  2036007.18            6480
1     fleet  318.732198     155.520  2051679.16            6437
2    public  317.826759     153.795  2050618.25            6452
3      taxi  313.799866     153.005  2032795.53            6478


In [6]:
top_customers = customer_clv.sort_values('clv', ascending=False).head(10)
print(top_customers[['customer_id', 'user_type', 'frequency', 'clv']])

     customer_id user_type  frequency      clv
1150  CUST_00288    public         84  2540.22
178   CUST_00045    public         86  2523.29
299   CUST_00075      taxi         92  2511.04
1457  CUST_00365     fleet         91  2440.76
308   CUST_00078  delivery         90  2435.37
809   CUST_00203     fleet         84  2403.82
1414  CUST_00354    public         90  2396.79
859   CUST_00215      taxi         83  2392.35
1238  CUST_00310    public         90  2374.86
817   CUST_00205     fleet         78  2370.42


In [7]:
segment_clv.to_csv('/content/drive/MyDrive/Cadetx /segment_clv.csv', index=False)
top_customers.to_csv('/content/drive/MyDrive/Cadetx /top_customers_clv.csv', index=False)

print("Saved!")

Saved!
